In [0]:
%pip install seaborn

In [0]:
# ============================================================
# CÉLULA 1 — Setup e carregamento (versão robusta)
# Notebook 06: modelo final e priorização de municípios
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# --- Carregar a base Gold (fonte oficial) ---
pdf = spark.table("default.gold_indicador_municipio").toPandas()

print(f"Shape original: {pdf.shape}")
print(f"Municípios únicos: {pdf['id_municipio'].nunique()}")
print(f"\nColunas ({len(pdf.columns)}):")
for c in sorted(pdf.columns):
    print(f"  - {c}")

In [0]:
# ============================================================
# CÉLULA 2 — Features de contexto com separação temporal
# Features vêm de 2023 (ano anterior) | Target vem de 2024
# Elimina o vazamento: gap_pp e taxa_observada do ano do target
# NÃO entram como features.
# ============================================================
import numpy as np

ANO_FEATURES = 2023
ANO_TARGET   = 2024

# --- UF a partir do código IBGE ---
pdf["codigo_uf"] = pdf["id_municipio"].astype(str).str.zfill(7).str[:2]

# --- Base de features (2023) ---
pdf_feat = pdf[pdf["ano"] == ANO_FEATURES].copy()

FEAT_ANT = ["taxa_observada", "percentual_participacao", "qtd_redes", "media_portugues"]
pdf_feat = pdf_feat[["id_municipio", "codigo_uf"] + FEAT_ANT].rename(
    columns={c: f"{c}_ant" for c in FEAT_ANT}
)

# --- Contexto federativo calculado em 2023 ---
uf_ant = (
    pdf_feat.groupby("codigo_uf")["taxa_observada_ant"]
    .mean().rename("taxa_uf_ant").reset_index()
)
pdf_feat["posicao_relativa_uf"] = (
    pdf_feat.groupby("codigo_uf")["taxa_observada_ant"].rank(pct=True, ascending=True)
)

# --- Target e meta de 2024 ---
pdf_tgt = pdf[pdf["ano"] == ANO_TARGET][["id_municipio", "status_meta", "meta_do_ano"]].copy()
pdf_tgt["target"] = (pdf_tgt["status_meta"].astype(str).str.strip().str.lower() == "atingiu").astype(int)
pdf_tgt = pdf_tgt.rename(columns={"meta_do_ano": "meta_target"})

# --- Montar base final ---
pdf_v2 = (
    pdf_feat
    .merge(uf_ant, on="codigo_uf", how="left")
    .merge(pdf_tgt, on="id_municipio", how="inner")
)
pdf_v2["meta_vs_uf"] = pdf_v2["meta_target"] / pdf_v2["taxa_uf_ant"]

FEATURES_V2 = [
    "taxa_observada_ant",
    "percentual_participacao_ant",
    "qtd_redes_ant",
    "media_portugues_ant",
    "meta_target",
    "taxa_uf_ant",
    "posicao_relativa_uf",
    "meta_vs_uf",
]

# --- Filtrar registros válidos ---
pdf_model = pdf_v2[pdf_v2["status_meta"].isin(["atingiu", "abaixo"])].copy()
pdf_model = pdf_model.dropna(subset=FEATURES_V2 + ["target"])

print(f"Registros: {len(pdf_model)}")
print(f"Municípios: {pdf_model['id_municipio'].nunique()}")
print(f"UFs: {pdf_model['codigo_uf'].nunique()}")
print(f"\nDistribuição do target:")
print(pdf_model["target"].value_counts(normalize=True).round(4))
print(f"\nNulos por feature:")
print(pdf_model[FEATURES_V2].isna().sum())

In [0]:
# ============================================================
# CÉLULA 3 — Retreino do modelo final (sem vazamento)
# ============================================================
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

modelo_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=30,
        random_state=42,
        n_jobs=-1,
    )),
])

X = pdf_model[FEATURES_V2]
y = pdf_model["target"]

modelo_final.fit(X, y)

print(f"Modelo treinado: {len(X)} registros, {len(FEATURES_V2)} features")
print(f"Features: {FEATURES_V2}")

In [0]:
# ============================================================
# CÉLULA 4 — Scoring e validação por faixa de risco
# ============================================================
pdf_model["prob_abaixo"] = modelo_final.predict_proba(pdf_model[FEATURES_V2])[:, 0]

def faixa_risco(p):
    if p >= 0.80:
        return "Crítico"
    elif p >= 0.65:
        return "Alto"
    elif p >= 0.50:
        return "Médio"
    else:
        return "Baixo"

pdf_model["faixa_risco"] = pdf_model["prob_abaixo"].apply(faixa_risco)

resumo = (
    pdf_model.groupby("faixa_risco")
    .agg(
        municipios=("id_municipio", "count"),
        prob_media=("prob_abaixo", "mean"),
    )
    .reindex(["Crítico", "Alto", "Médio", "Baixo"])
    .round(4)
)
resumo["pct"] = (resumo["municipios"] / len(pdf_model) * 100).round(2)

print("=" * 65)
print("DISTRIBUIÇÃO POR FAIXA DE RISCO")
print("=" * 65)
print(resumo)
print(f"\nTotal: {int(resumo['municipios'].sum())} municípios")

print("\n" + "=" * 65)
print("TAXA REAL DE 'ABAIXO' POR FAIXA (validação in-sample)")
print("=" * 65)
validacao = (
    pdf_model.groupby("faixa_risco")
    .agg(
        municipios=("id_municipio", "count"),
        pct_real_abaixo=("target", lambda s: (s == 0).mean()),
    )
    .reindex(["Crítico", "Alto", "Médio", "Baixo"])
    .round(4)
)
print(validacao)

In [0]:
# ============================================================
# CÉLULA 5 — Cruzamento entre faixa de risco e magnitude do gap
# ============================================================
gap_2024 = pdf[pdf["ano"] == 2024][["id_municipio", "gap_pp"]].rename(
    columns={"gap_pp": "gap_pp_real"}
)

pdf_model = pdf_model.merge(gap_2024, on="id_municipio", how="left")

def faixa_gap(g):
    if pd.isna(g):
        return "Sem dado"
    elif g >= -2:
        return "Gap pequeno (até 2 p.p.)"
    elif g >= -10:
        return "Gap médio (2 a 10 p.p.)"
    else:
        return "Gap crítico (> 10 p.p.)"

pdf_model["faixa_gap"] = pdf_model["gap_pp_real"].apply(faixa_gap)

cruzamento = pd.crosstab(
    pdf_model["faixa_risco"],
    pdf_model["faixa_gap"],
    margins=True,
    margins_name="Total",
).reindex(["Crítico", "Alto", "Médio", "Baixo", "Total"])

print("=" * 75)
print("CRUZAMENTO: FAIXA DE RISCO × MAGNITUDE DO GAP (2024)")
print("=" * 75)
print(cruzamento)

print("\n" + "=" * 75)
print("PERFIL DOS MUNICÍPIOS PRIORITÁRIOS (Crítico + Alto)")
print("=" * 75)
prioritarios = pdf_model[pdf_model["faixa_risco"].isin(["Crítico", "Alto"])]
print(f"Total: {len(prioritarios)} municípios")
print(f"Gap médio: {prioritarios['gap_pp_real'].mean():.2f} p.p.")
print(f"Taxa média 2023: {prioritarios['taxa_observada_ant'].mean():.2f}")
print(f"Meta média 2024: {prioritarios['meta_target'].mean():.2f}")
print(f"\nDistribuição por magnitude do gap:")
print(prioritarios["faixa_gap"].value_counts())

In [0]:
# ============================================================
# CÉLULA 6 — Top 20 municípios prioritários
# ============================================================
top20 = (
    pdf_model[pdf_model["faixa_risco"].isin(["Crítico", "Alto"])]
    .sort_values(["prob_abaixo", "gap_pp_real"], ascending=[False, True])
    .head(20)
    [["id_municipio", "codigo_uf", "prob_abaixo", "gap_pp_real",
      "taxa_observada_ant", "meta_target", "faixa_risco", "faixa_gap"]]
    .reset_index(drop=True)
)
top20.index = top20.index + 1
top20.index.name = "Rank"

print("=" * 90)
print("TOP 20 MUNICÍPIOS PRIORITÁRIOS")
print("=" * 90)
print(top20.round(4))

In [0]:
# ============================================================
# CÉLULA 7 — Concentração de risco por UF
# ============================================================
por_uf = (
    pdf_model.groupby("codigo_uf")
    .agg(
        municipios=("id_municipio", "count"),
        criticos=("faixa_risco", lambda s: (s == "Crítico").sum()),
        altos=("faixa_risco", lambda s: (s == "Alto").sum()),
        prob_media=("prob_abaixo", "mean"),
        gap_medio=("gap_pp_real", "mean"),
    )
    .round(4)
)
por_uf["prioritarios"] = por_uf["criticos"] + por_uf["altos"]
por_uf["pct_prioritario"] = (por_uf["prioritarios"] / por_uf["municipios"] * 100).round(2)
por_uf = por_uf.sort_values("pct_prioritario", ascending=False)

print("=" * 90)
print("CONCENTRAÇÃO DE RISCO POR UF")
print("=" * 90)
print(por_uf)

In [0]:
# ============================================================
# CÉLULA 8 — Exportação da lista de priorização (Delta)
# DBFS público desabilitado: gravamos apenas como tabela Delta
# ============================================================
saida = pdf_model[[
    "id_municipio", "codigo_uf", "prob_abaixo", "faixa_risco",
    "gap_pp_real", "faixa_gap", "taxa_observada_ant",
    "meta_target", "target",
]].sort_values("prob_abaixo", ascending=False).reset_index(drop=True)

# --- Delta (fonte oficial) ---
(
    spark.createDataFrame(saida)
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("default.gold_priorizacao_municipios")
)

print(f"Exportados {len(saida)} municípios")
print(f"Prioritários (Crítico + Alto): {(saida['faixa_risco'].isin(['Crítico','Alto'])).sum()}")
print(f"\nDistribuição por faixa:")
print(saida["faixa_risco"].value_counts().reindex(["Crítico","Alto","Médio","Baixo"]))
print(f"\nTabela: default.gold_priorizacao_municipios")

In [0]:
# ============================================================
# CÉLULA 9 — Conclusão do notebook 06
# ============================================================
print("=" * 70)
print("NOTEBOOK 06 — MODELO FINAL E PRIORIZAÇÃO DE MUNICÍPIOS")
print("=" * 70)

print(f"\n1. BASE")
print(f"   Registros modelados: {len(pdf_model)}")
print(f"   Features: {len(FEATURES_V2)} (4 históricas + 4 de contexto)")
print(f"   Separação temporal: features 2023 -> target 2024")

print(f"\n2. DESEMPENHO POR FAIXA (validação in-sample)")
for faixa in ["Crítico", "Alto", "Médio", "Baixo"]:
    sub = pdf_model[pdf_model["faixa_risco"] == faixa]
    if len(sub) > 0:
        pct = (sub["target"] == 0).mean() * 100
        print(f"   {faixa:8s}: {len(sub):5d} municípios | {pct:5.2f}% realmente abaixo")

print(f"\n3. LISTA PRIORITÁRIA")
prio = pdf_model[pdf_model["faixa_risco"].isin(["Crítico", "Alto"])]
print(f"   Total: {len(prio)} municípios ({len(prio)/len(pdf_model)*100:.1f}% da base)")
print(f"   Gap médio: {prio['gap_pp_real'].mean():.2f} p.p.")
print(f"   Em gap crítico (>10 p.p.): {(prio['faixa_gap']=='Gap crítico (> 10 p.p.)').sum()}")

print(f"\n4. CONCENTRAÇÃO GEOGRÁFICA")
for uf, row in por_uf.head(2).iterrows():
    print(f"   UF {uf}: {int(row['prioritarios'])} prioritários ({row['pct_prioritario']:.1f}% dos municípios)")

print(f"\n5. CONCLUSÃO")
print(f"   O modelo identifica 905 municípios prioritários com 87,3% de acerto real.")
print(f"   RS e BA concentram 69,5% da lista — a política deve ser estadual, não nacional.")
print(f"   A separação temporal elimina vazamento: features de 2023, target de 2024.")
print("=" * 70)

In [0]:
# ============================================================
# CÉLULA 10 — Visualizações para o relatório
# ============================================================
import matplotlib.pyplot as plt

UF_SIGLA = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL",
    "28": "SE", "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Gráfico 1: distribuição por faixa de risco ---
ordem = ["Crítico", "Alto", "Médio", "Baixo"]
cores = ["#B71C1C", "#EF6C00", "#F9A825", "#2E7D32"]
contagem = pdf_model["faixa_risco"].value_counts().reindex(ordem)

axes[0].bar(ordem, contagem.values, color=cores)
axes[0].set_title("Municípios por faixa de risco")
axes[0].set_ylabel("Quantidade de municípios")
for i, v in enumerate(contagem.values):
    axes[0].text(i, v + 40, f"{v:,}".replace(",", "."), ha="center", fontweight="bold")

# --- Gráfico 2: prioritários por UF (top 10) ---
top_ufs = por_uf.sort_values("prioritarios", ascending=False).head(10)
top_ufs = top_ufs.sort_values("prioritarios", ascending=True)  # menor no topo do barh
rotulos = [UF_SIGLA.get(str(uf), str(uf)) for uf in top_ufs.index]

axes[1].barh(rotulos, top_ufs["prioritarios"], color="#1565C0")
axes[1].set_title("Municípios prioritários por UF (top 10)")
axes[1].set_xlabel("Prioritários (Crítico + Alto)")
for i, v in enumerate(top_ufs["prioritarios"]):
    axes[1].text(v + 5, i, str(int(v)), va="center", fontweight="bold")

plt.tight_layout()
plt.show()

In [0]:
print("Registros:", len(pdf_model))
print("Prioritários:", (pdf_model["faixa_risco"].isin(["Crítico", "Alto"])).sum())
print("Tabela Delta:", spark.table("default.gold_priorizacao_municipios").count())